# M06 · Axis supervision - following error, collision, dead-man and homing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/04_control_motion/M06_axis_supervision_following_error_collision_dead_.ipynb)

**Track:** Control and motion · **Skill axis (sheet):** Control and motion · **Time:** about 60 min

**Skill Hub concepts:** Following error and homing; Current-based collision detection; Dead-man switch and velocity limiting

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("M06")

## Map: the checks that wrap every motion controller

| Check | Signal | Trips when | Response |
|---|---|---|---|
| Velocity limit | commanded position change per cycle | always active | clamp the step to $v_{max}\,\Delta t$ |
| Following error | $\lvert\text{commanded} - \text{actual}\rvert$ | above a limit (jam, broken coupling, weak drive) | latch fault, disable |
| Collision | motor current $i = \tau_{joint} / (N K_t)$ | above a threshold for N consecutive cycles | latch fault, disable |
| Dead-man (enabling) switch | operator holds a button | released | hold position, accept no new motion |

Faults **latch**: the axis stays disabled until an explicit reset, even if the condition clears, so an intermittent
fault cannot restart motion by itself. On reset, set the command to the actual position (bumpless).

**Homing** gives a repeatable zero:
```
 SEEK (fast toward switch) ──switch pressed──> BACKOFF (slow away) ──switch released──> APPROACH (slow toward)
     ──switch pressed──> zero = position here ──> DONE           any state ──timeout──> FAILED
```
The slow second approach makes the trigger point independent of the fast speed (and an encoder index pulse refines it).

## Task 1: the supervisor

In [ ]:
class AxisSupervisor:
    def __init__(self, ferr_max, i_limit, i_cycles, v_max, dt):
        self.ferr_max, self.i_limit, self.i_cycles, self.v_max, self.dt = ferr_max, i_limit, i_cycles, v_max, dt
        self.fault, self.cmd, self.over = None, 0.0, 0

    def update(self, target, actual, current, deadman):
        """Return (enable, command)."""
        raise NotImplementedError  # TODO

    def reset(self, actual):
        raise NotImplementedError  # TODO

lab.check("supervisor", AxisSupervisor)

In [ ]:
%%writefile m06_supervisor.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/m06.h
// Grade it with lab.check("supervisor", "m06_supervisor.c"). For C++, name the file .cpp in both places.
#include "m06.h"

void axis_init(AxisSupervisor *s, double ferr_max, double i_limit, int i_cycles, double v_max, double dt) {
    s->ferr_max = ferr_max; s->i_limit = i_limit; s->i_cycles = i_cycles; s->v_max = v_max; s->dt = dt;
    s->fault = FAULT_NONE; s->cmd = 0; s->over = 0;
}

bool axis_update(AxisSupervisor *s, double target, double actual, double current, bool deadman, double *command) {
    (void)s; (void)target; (void)current; (void)deadman;
    *command = actual;
    return false; /* TODO: latch faults, rate limit with the dead-man, count over-current cycles, following error */
}

void axis_reset(AxisSupervisor *s, double actual) { (void)s; (void)actual; /* TODO: bumpless */ }

In [ ]:
# lab.check("supervisor", "m06_supervisor.c")  # uncomment after writing the C/C++ cell above

## Task 2: the homing state machine
`step(switch, position)` is called every cycle and returns the velocity command. Seek in the negative direction at
`v_fast`, back off at `+v_slow`, approach at `-v_slow`; on the final press store `self.zero = position` and stop.

In [ ]:
class Homing:
    def __init__(self, v_fast, v_slow, timeout_s, dt):
        self.v_fast, self.v_slow, self.timeout, self.dt = v_fast, v_slow, timeout_s, dt
        self.state, self.t, self.zero = "SEEK", 0.0, None

    def step(self, switch, position):
        raise NotImplementedError  # TODO

lab.check("homing", Homing)

In [ ]:
%%writefile m06_homing.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/m06.h
// Grade it with lab.check("homing", "m06_homing.c"). For C++, name the file .cpp in both places.
#include "m06.h"

void homing_init(Homing *h, double v_fast, double v_slow, double timeout_s, double dt) {
    h->v_fast = v_fast; h->v_slow = v_slow; h->timeout = timeout_s; h->dt = dt;
    h->state = HOME_SEEK; h->t = 0; h->zero = 0;
}

double homing_step(Homing *h, bool switch_active, double position) {
    (void)h; (void)switch_active; (void)position;
    return 0.0; /* TODO: SEEK fast -> BACKOFF slow off the switch -> APPROACH slow -> DONE; timeout -> FAILED */
}

In [ ]:
# lab.check("homing", "m06_homing.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** Joint torque 2 N·m through a 50:1 gearbox with $K_t = 0.1$ N·m/A. Motor current in amperes?

**M2 (Apply).** Dead-man released at 0.5 m/s; the axis decelerates at 2 m/s$^2$. Stopping distance in meters?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Evaluate).** A collision fault clears itself when the current drops. Should motion resume automatically?
- a) yes, it saves time
- b) no: latch the fault and require an explicit reset, so an intermittent or still-present hazard cannot restart motion
- c) only if the following error is small

**K2 (Analyze).** Why home in two passes (fast, then slow) and how does an encoder index pulse improve repeatability?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `m06.h`).
3. In `roboprep/checks/m06.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()